In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — INSTALL  (run once, wait for it to finish)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
!pip install -q \
    langsmith \
    langchain \
    langchain-core \
    langchain-openai \
    openai \
    presidio-analyzer \
    presidio-anonymizer

!python -m spacy download en_core_web_lg -q
print("✅ Install complete — now RESTART RUNTIME (Runtime → Restart runtime)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.5 which is incompatible.
pyopenssl 24.2.1 requires cryptography<44,>=41.0.5, but you have cryptography 46.0.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 3.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the packa

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — API KEYS  (run this FIRST after every restart)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

# ── Paste your keys here ─────────────────────────────────────
LANGSMITH_KEY = ""   # smith.langchain.com → Settings → API Keys → Create (Personal Access Token, Full Access)
OPENAI_KEY    = ""       # platform.openai.com → API keys

# ── Set every env var both SDKs check ───────────────────────
for k, v in {
    "LANGCHAIN_API_KEY":    LANGSMITH_KEY,
    "LANGSMITH_API_KEY":    LANGSMITH_KEY,
    "OPENAI_API_KEY":       OPENAI_KEY,
    "LANGCHAIN_TRACING_V2": "true",
    "LANGSMITH_TRACING":    "true",
    "LANGCHAIN_PROJECT":    "production-pipeline-v1",
    "LANGSMITH_PROJECT":    "production-pipeline-v1",
    "LANGCHAIN_ENDPOINT":   "https://api.smith.langchain.com",
    "LANGSMITH_ENDPOINT":   "https://api.smith.langchain.com",
}.items():
    os.environ[k] = v

print(f"✅ Keys set | LangSmith: {LANGSMITH_KEY[:12]}...{LANGSMITH_KEY[-4:]} | len={len(LANGSMITH_KEY)}")

✅ Keys set | LangSmith: lsv2_pt_eb10...d400 | len=51


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — PERMISSION CHECK  (all 4 must show ✅ before continuing)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import requests, json, uuid

key     = os.environ["LANGSMITH_API_KEY"]
headers = {"x-api-key": key, "Content-Type": "application/json"}
base    = "https://api.smith.langchain.com"
OK      = (200, 201, 202)

# READ
for label, url in [
    ("READ  /sessions", f"{base}/sessions?limit=1"),
    ("READ  /info",     f"{base}/info"),
]:
    r = requests.get(url, headers=headers)
    print(f"{'✅' if r.status_code in OK else '❌  → regenerate key!'} {label:25s} {r.status_code}")

# WRITE runs
r = requests.post(f"{base}/runs", headers=headers, data=json.dumps({
    "id": str(uuid.uuid4()), "name": "perm-test", "run_type": "chain",
    "inputs": {"x": 1}, "start_time": "2025-01-01T00:00:00Z",
    "end_time": "2025-01-01T00:00:01Z", "outputs": {"y": 1},
}))
print(f"{'✅' if r.status_code in OK else '❌  → regenerate key with Full Access!'} {'WRITE /runs':25s} {r.status_code}")

# WRITE feedback
r = requests.post(f"{base}/feedback", headers=headers,
                  data=json.dumps({"run_id": str(uuid.uuid4()), "key": "test", "score": 1}))
print(f"{'✅' if r.status_code in OK else '❌  → regenerate key with Full Access!'} {'WRITE /feedback':25s} {r.status_code}")

# Get handle
r = requests.get(f"{base}/current-tenant", headers=headers)
handle = r.json().get("handle", "") if r.status_code in OK else ""
os.environ["LANGSMITH_HANDLE"] = handle
print(f"\n✅ LangSmith handle: '{handle}'")
print("━━━  All 4 green? Proceed. Any red? Regenerate key at smith.langchain.com  ━━━")

✅ READ  /sessions           200
✅ READ  /info               200
✅ WRITE /runs               202
✅ WRITE /feedback           202

✅ LangSmith handle: ''
━━━  All 4 green? Proceed. Any red? Regenerate key at smith.langchain.com  ━━━


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — IMPORTS & CLIENT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from langsmith import Client, traceable, evaluate
from langsmith.run_helpers import get_current_run_tree
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig
from datetime import datetime, timezone
import uuid, json, time

# Always pass key explicitly — never rely on env var timing
ls = Client(
    api_url="https://api.smith.langchain.com",
    api_key=os.environ["LANGSMITH_API_KEY"],
)

print(f"✅ Client ready")
print(f"✅ Projects: {[p.name for p in ls.list_projects()][:5]}")

✅ Client ready
✅ Projects: ['default']


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — PROMPT VERSIONS
# Push v1/v2/v3 to LangSmith Hub — creates full version history
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from langchain_core.prompts import ChatPromptTemplate

prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer concisely."),
    ("human", "{question}"),
])

prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert assistant.\n"
        "Think step by step.\n"
        "Format: REASONING: ... | ANSWER: ..."
    )),
    ("human", "{question}"),
])

prompt_v3 = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an expert assistant specialized in {domain}.\n"
        "Think step by step. Be precise.\n"
        "Format: REASONING: ... | ANSWER: ... | CONFIDENCE: high/medium/low"
    )),
    ("human", "{question}"),
])

PROMPT_REGISTRY = {
    "prompt_v1": prompt_v1,
    "prompt_v2": prompt_v2,
    "prompt_v3": prompt_v3,
}

# Push to Hub
handle = os.environ.get("LANGSMITH_HANDLE", "")
if handle:
    PROMPT_NAME = f"{handle}/my-assistant-prompt"
    for label, prompt in PROMPT_REGISTRY.items():
        try:
            ls.push_prompt(PROMPT_NAME, object=prompt)
            print(f"✅ Pushed {label} → {PROMPT_NAME}")
        except Exception as e:
            print(f"⚠️  {label}: {e}")

    # List all versions
    try:
        commits = list(ls.list_prompt_commits(PROMPT_NAME))
        print(f"\n📋 {len(commits)} versions in Hub:")
        for c in commits:
            print(f"   {str(c.manifest_id)[:12]}... | {c.created_at}")
    except Exception as e:
        print(f"⚠️  list commits: {e}")

    # Pull latest back
    try:
        pulled = ls.pull_prompt(PROMPT_NAME)
        print(f"\n✅ Pulled latest: {type(pulled)}")
    except Exception as e:
        print(f"⚠️  pull: {e}")
else:
    print("⚠️  No handle — prompts available locally only")
    print("   Hardcode: os.environ['LANGSMITH_HANDLE'] = 'your-username'")

print("\n📊 UI: smith.langchain.com → Hub → my-assistant-prompt → version history")

⚠️  No handle — prompts available locally only
   Hardcode: os.environ['LANGSMITH_HANDLE'] = 'your-username'

📊 UI: smith.langchain.com → Hub → my-assistant-prompt → version history


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — DEPLOYMENT VERSION MATRIX
# Every combination of system + model + prompt + retriever
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DEPLOYMENTS = {
    "system_v1": {
        "model":       "gpt-4o-mini",
        "prompt":      "prompt_v2",
        "retriever":   "embeddings_v1",
        "reranker":    "none",
        "temperature": 0.2,
    },
    "system_v2": {
        "model":       "gpt-4o",
        "prompt":      "prompt_v3",
        "retriever":   "embeddings_v2",
        "reranker":    "cohere_v2",
        "temperature": 0.0,
    },
    "system_v3_experiment": {
        "model":       "gpt-4o",
        "prompt":      "prompt_v3",
        "retriever":   "embeddings_v3_colbert",
        "reranker":    "cohere_v3",
        "temperature": 0.1,
    },
}

print("✅ Deployment matrix:")
print(f"  {'NAME':25s}  {'MODEL':12s}  {'PROMPT':10s}  {'RETRIEVER':20s}  RERANKER")
print("  " + "─"*80)
for name, cfg in DEPLOYMENTS.items():
    print(f"  {name:25s}  {cfg['model']:12s}  {cfg['prompt']:10s}  {cfg['retriever']:20s}  {cfg['reranker']}")

✅ Deployment matrix:
  NAME                       MODEL         PROMPT      RETRIEVER             RERANKER
  ────────────────────────────────────────────────────────────────────────────────
  system_v1                  gpt-4o-mini   prompt_v2   embeddings_v1         none
  system_v2                  gpt-4o        prompt_v3   embeddings_v2         cohere_v2
  system_v3_experiment       gpt-4o        prompt_v3   embeddings_v3_colbert  cohere_v3


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — PII ENGINE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_lg"}],
})
analyzer   = AnalyzerEngine(nlp_engine=provider.create_engine())
anonymizer = AnonymizerEngine()

PII_ENTITIES = ["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "IP_ADDRESS", "LOCATION", "DATE_TIME"]
ANON_OPS = {
    "PERSON":        OperatorConfig("replace", {"new_value": "<PERSON>"}),
    "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
    "PHONE_NUMBER":  OperatorConfig("replace", {"new_value": "<PHONE>"}),
    "IP_ADDRESS":    OperatorConfig("replace", {"new_value": "<IP>"}),
    "LOCATION":      OperatorConfig("replace", {"new_value": "<LOCATION>"}),
    "DATE_TIME":     OperatorConfig("replace", {"new_value": "<DATE>"}),
}

# Quick smoke test
_test = analyzer.analyze(text="John lives in Paris", entities=PII_ENTITIES, language="en")
print(f"✅ PII engine ready | smoke test found {len(_test)} entities: {[r.entity_type for r in _test]}")

✅ PII engine ready | smoke test found 2 entities: ['PERSON', 'LOCATION']


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — PIPELINE FUNCTIONS
# client=ls passed explicitly to every @traceable → fixes 403
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

@traceable(name="pii-detection", client=ls)
def detect_pii(text: str) -> dict:
    results = analyzer.analyze(text=text, entities=PII_ENTITIES, language="en")
    return {
        "raw":      results,
        "count":    len(results),
        "entities": [r.entity_type for r in results],
    }

@traceable(name="anonymization", client=ls)
def anonymize(text: str, pii_results: list) -> str:
    return anonymizer.anonymize(
        text=text, analyzer_results=pii_results, operators=ANON_OPS
    ).text

@traceable(name="llm-call", client=ls)
def call_llm(prompt: str, model: str, template, domain: str = "general") -> str:
    llm   = ChatOpenAI(model=model, temperature=0)
    chain = template | llm | StrOutputParser()
    inp   = {"question": prompt}
    if "{domain}" in str(template):
        inp["domain"] = domain
    return chain.invoke(inp)

@traceable(
    name="full-production-pipeline",
    project_name="production-pipeline-v1",
    tags=["production"],
    client=ls,
)
def run_pipeline(
    raw_input:       str,
    deployment_name: str = "system_v1",
    user_id:         str = "anonymous",
    session_id:      str = None,
    domain:          str = "general",
) -> dict:
    cfg        = DEPLOYMENTS[deployment_name]
    session_id = session_id or str(uuid.uuid4())

    # Stamp full version snapshot on this trace
    rt = get_current_run_tree()
    if rt:
        rt.metadata.update({
            "deployment":        deployment_name,
            "model_version":     cfg["model"],
            "prompt_version":    cfg["prompt"],
            "retriever_version": cfg["retriever"],
            "reranker_version":  cfg["reranker"],
            "temperature":       cfg["temperature"],
            "user_id":           user_id,
            "session_id":        session_id,
            "domain":            domain,
            "environment":       "production",
            "timestamp":         datetime.now(timezone.utc).isoformat(),
        })
        rt.tags = (rt.tags or []) + [
            deployment_name,
            cfg["model"],
            cfg["prompt"],
            cfg["retriever"],
            f"user:{user_id}",
        ]

    pii_out    = detect_pii(raw_input)
    safe_input = anonymize(raw_input, pii_out["raw"])
    answer     = call_llm(
        safe_input, cfg["model"],
        PROMPT_REGISTRY[cfg["prompt"]], domain
    )

    return {
        "safe_input":   safe_input,
        "answer":       answer,
        "pii_found":    pii_out["count"],
        "pii_entities": pii_out["entities"],
        "deployment":   deployment_name,
        "session_id":   session_id,
        "run_id":       str(rt.id) if rt else None,
    }

print("✅ Pipeline functions defined")

✅ Pipeline functions defined


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 9 — RUN ALL 3 DEPLOYMENTS + COLLECT RUN IDS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
question   = "My name is Alice Smith, I live in London. Explain RAG to me."
session_id = str(uuid.uuid4())
run_ids    = {}

for deployment in DEPLOYMENTS:
    print(f"\n🚀 {deployment}")
    result = run_pipeline(
        raw_input       = question,
        deployment_name = deployment,
        user_id         = "user_42",
        session_id      = session_id,
        domain          = "AI & Machine Learning",
    )
    run_ids[deployment] = result["run_id"]
    print(f"   PII:    {result['pii_found']} → {result['pii_entities']}")
    print(f"   Safe:   {result['safe_input']}")
    print(f"   Answer: {result['answer'][:120]}...")
    print(f"   Run ID: {result['run_id']}")

print(f"\n✅ All 3 runs done | Session: {session_id}")
print("📊 smith.langchain.com → Projects → production-pipeline-v1")
print("   Click any run → Metadata tab → see full version snapshot")


🚀 system_v1
   PII:    2 → ['PERSON', 'LOCATION']
   Safe:   My name is <PERSON>, I live in <LOCATION>. Explain RAG to me.
   Answer: REASONING: RAG stands for Retrieval-Augmented Generation, which is a technique used in natural language processing and m...
   Run ID: 019cd946-f0b8-7183-8b54-2fd7200f5300

🚀 system_v2
   PII:    2 → ['PERSON', 'LOCATION']
   Safe:   My name is <PERSON>, I live in <LOCATION>. Explain RAG to me.
   Answer: REASONING: RAG stands for Retrieval-Augmented Generation, a framework used in natural language processing and machine le...
   Run ID: 019cd947-04f5-7e21-a687-d8d05603c785

🚀 system_v3_experiment
   PII:    2 → ['PERSON', 'LOCATION']
   Safe:   My name is <PERSON>, I live in <LOCATION>. Explain RAG to me.
   Answer: REASONING: RAG stands for Retrieval-Augmented Generation, a framework used in natural language processing and machine le...
   Run ID: 019cd947-1612-7a00-bb36-76a0a83a10de

✅ All 3 runs done | Session: f8dc135b-7d88-4afc-93fb-0a450e79daa3
📊

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 11 — SESSION REPLAY
# All turns share one session_id → see full conversation in UI
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SESSION_ID = str(uuid.uuid4())

@traceable(name="chat-turn", project_name="production-pipeline-v1", client=ls)
def chat_turn(question: str, turn: int) -> str:
    rt = get_current_run_tree()
    if rt:
        rt.metadata.update({
            "session_id": SESSION_ID,
            "user_id":    "user_42",
            "turn":       turn,
        })
        rt.tags = (rt.tags or []) + [f"turn:{turn}", "chat"]
    llm   = ChatOpenAI(model="gpt-4o-mini")
    chain = prompt_v2 | llm | StrOutputParser()
    return chain.invoke({"question": question})

turns = [
    "What is RAG?",
    "How does the retriever component work?",
    "Which embedding models work best with RAG?",
]

print(f"Session: {SESSION_ID}\n")
for i, q in enumerate(turns, 1):
    resp = chat_turn(q, turn=i)
    print(f"Turn {i}: {resp[:100]}...")

print(f"\n✅ Filter in UI: metadata.session_id = {SESSION_ID}")

Session: fb9d3118-e7cf-41e0-83c0-2ce98b1fbae0

Turn 1: REASONING: RAG usually refers to "Red, Amber, Green," a color-coded system commonly used in project ...
Turn 2: REASONING: The retriever component is often part of a larger information retrieval or machine learni...
Turn 3: REASONING: Retrieval-Augmented Generation (RAG) models combine the strengths of retrieval-based and ...

✅ Filter in UI: metadata.session_id = fb9d3118-e7cf-41e0-83c0-2ce98b1fbae0


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 12 — GOLDEN DATASET
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATASET_NAME = "rag-qa-golden-set-v1"

existing = [d.name for d in ls.list_datasets()]
if DATASET_NAME in existing:
    dataset = next(d for d in ls.list_datasets() if d.name == DATASET_NAME)
    print(f"✅ Dataset exists: {dataset.id}")
else:
    dataset = ls.create_dataset(DATASET_NAME, description="Golden QA pairs for RAG eval")
    ls.create_examples(
        inputs=[
            {"question": "What is RAG?"},
            {"question": "What is a vector database?"},
            {"question": "What is LangSmith used for?"},
            {"question": "Difference between fine-tuning and RAG?"},
            {"question": "What is chunking in RAG pipelines?"},
        ],
        outputs=[
            {"answer": "RAG is Retrieval Augmented Generation — it grounds LLM responses in external documents retrieved at inference time."},
            {"answer": "A vector database stores high-dimensional embeddings and enables fast similarity search."},
            {"answer": "LangSmith is an observability and evaluation platform for LLM applications by LangChain."},
            {"answer": "Fine-tuning updates model weights. RAG retrieves documents at inference time without changing weights."},
            {"answer": "Chunking splits documents into smaller pieces so they fit in the context window and improve retrieval precision."},
        ],
        dataset_id=dataset.id,
    )
    print(f"✅ Dataset created with 5 examples: {dataset.id}")

print(f"📊 UI → Datasets → {DATASET_NAME}")

✅ Dataset created with 5 examples: 2249a262-324f-4eb7-9aa5-c3e7f26d33b1
📊 UI → Datasets → rag-qa-golden-set-v1


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 13 — AUTOMATED EVALUATION
# 3 evaluators: keyword overlap · LLM judge · length check
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# What we're testing
def pipeline_under_test(inputs: dict) -> dict:
    llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    chain = prompt_v2 | llm | StrOutputParser()
    return {"answer": chain.invoke({"question": inputs["question"]})}

# Evaluator 1 — keyword overlap
def keyword_overlap(outputs: dict, reference_outputs: dict) -> dict:
    pred  = outputs.get("answer", "").lower()
    ref   = reference_outputs.get("answer", "").lower()
    words = [w for w in ref.split() if len(w) > 5]
    score = round(sum(1 for w in words if w in pred) / len(words), 3) if words else 0.0
    return {"key": "keyword_overlap", "score": score}

# Evaluator 2 — LLM as judge
def llm_judge(outputs: dict, reference_outputs: dict) -> dict:
    pred    = outputs.get("answer", "")
    ref     = reference_outputs.get("answer", "")
    judge   = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    verdict = judge.invoke(
        f"Score PREDICTION vs REFERENCE from 0.0 to 1.0. Reply ONLY with a float.\n"
        f"REFERENCE:  {ref}\n"
        f"PREDICTION: {pred}"
    ).content.strip()
    try:    score = float(verdict)
    except: score = 0.0
    return {"key": "llm_judge", "score": score}

# Evaluator 3 — length sanity
def length_check(outputs: dict) -> dict:
    words = len(outputs.get("answer", "").split())
    return {"key": "length_ok", "score": 1.0 if 15 < words < 300 else 0.0}

print("🔬 Running evaluation against dataset...")
results = evaluate(
    pipeline_under_test,
    data=DATASET_NAME,
    evaluators=[keyword_overlap, llm_judge, length_check],
    experiment_prefix="system_v1-prompt_v2",
    metadata={
        "deployment":     "system_v1",
        "prompt_version": "prompt_v2",
        "model":          "gpt-4o-mini",
        "retriever":      "embeddings_v1",
    },
    max_concurrency=2,
    description="Baseline: system_v1 + prompt_v2 + gpt-4o-mini",
    client=ls,
)
print("✅ Evaluation complete")
print(f"📊 UI → Datasets → {DATASET_NAME} → Experiments tab → compare scores")

🔬 Running evaluation against dataset...
View the evaluation results for experiment: 'system_v1-prompt_v2-f2d33e1d' at:
https://smith.langchain.com/o/302c62e7-0cd9-4a40-ab47-5a2a9cc38523/datasets/2249a262-324f-4eb7-9aa5-c3e7f26d33b1/compare?selectedSessions=adfbd3fa-f849-4991-9aca-493174fad666




0it [00:00, ?it/s]

✅ Evaluation complete
📊 UI → Datasets → rag-qa-golden-set-v1 → Experiments tab → compare scores


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 14 — QUERY YOUR TRACES PROGRAMMATICALLY
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("📋 Last 10 top-level runs in production-pipeline-v1:\n")
runs = list(ls.list_runs(
    project_name="production-pipeline-v1",
    execution_order=1,
    limit=10,
))

print(f"  {'NAME':35s} {'STATUS':10s} {'DEPLOYMENT':25s} {'FB_AVG':8s}")
print("  " + "─"*82)
for r in runs:
    fb    = list(ls.list_feedback(run_ids=[r.id]))
    score = round(sum(f.score for f in fb if f.score is not None) / len(fb), 2) if fb else "—"
    meta  = r.metadata or {}
    print(f"  {r.name:35s} {r.status:10s} {meta.get('deployment','?'):25s} {str(score):8s}")

# Filter by deployment tag
print(f"\n📋 Runs tagged 'system_v2':")
for r in ls.list_runs(project_name="production-pipeline-v1",
                      filter='has(tags, "system_v2")', limit=5):
    print(f"  {r.id} | {r.name} | {r.status}")

📋 Last 10 top-level runs in production-pipeline-v1:

  NAME                                STATUS     DEPLOYMENT                FB_AVG  
  ──────────────────────────────────────────────────────────────────────────────────
  chat-turn                           success    ?                         —       
  chat-turn                           success    ?                         —       
  chat-turn                           success    ?                         —       
  full-production-pipeline            success    system_v3_experiment      —       
  full-production-pipeline            success    system_v2                 —       
  full-production-pipeline            success    system_v1                 —       

📋 Runs tagged 'system_v2':
  019cd947-04f5-7e21-a687-d8d05603c785 | full-production-pipeline | success
